# Tobin's Q og igangsatte boliger i Norge

## Teori
Tobin's Q for boligmarkedet er definert som:

$$Q_t = \frac{P_t^{\text{bolig}}}{C_t^{\text{bygging}}}$$

- **Q > 1**: Markedspris > byggekostnad → lønnsomt å bygge → igangsatte boliger øker  
- **Q < 1**: Markedspris < byggekostnad → ulønnsomt å bygge → igangsatte boliger faller

## Datakilder (SSB)
| Serie | SSB-tabell | Frekvens | Kilde |
|---|---|---|---|
| Boligprisindeks (brukte boliger) | 07221 | Kvartalsvis | [BPI](https://www.ssb.no/priser-og-prisindekser/statistikker/bpi/kvartal) |
| Byggekostnadsindeks (boligblokk) | 08655 | Månedlig | [BKI](https://www.ssb.no/priser-og-prisindekser/byggekostnadsindekser) |
| Igangsatte boliger | 06265 | Kvartalsvis | [Byggeareal](https://www.ssb.no/bygg-bolig-og-eiendom/statistikker/byggeareal) |

Alle indekser er normalisert til 2015=100.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SSB_BASE = 'https://data.ssb.no/api/v0/no/table/'

def ssb_query(table_id, query):
    """Henter data fra SSBs PxWebApi som json-stat2."""
    url = SSB_BASE + table_id
    payload = {**query, 'response': {'format': 'json-stat2'}}
    r = requests.post(url, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

def jsonstat2_to_df(js):
    """Konverterer json-stat2-respons til Pandas DataFrame."""
    dims = js['id']
    sizes = js['size']
    labels = {d: list(js['dimension'][d]['category']['label'].values()) for d in dims}
    keys   = {d: list(js['dimension'][d]['category']['label'].keys())   for d in dims}
    idx = pd.MultiIndex.from_product(labels.values(), names=dims)
    values = js['value']
    df = pd.DataFrame({'value': values}, index=idx).reset_index()
    return df

print('Biblioteker lastet.')

## 1. Boligprisindeks (tabell 07221)

In [ ]:
# Tabell 07221: Boligprisindeks etter boligtype og eierform (2015=100, kvartalsvis)
bpi_query = {
    'query': [
        {
            'code': 'Boligtype',
            'selection': {'filter': 'item', 'values': ['00']}  # 00 = Alle boligtyper
        },
        {
            'code': 'Eierform',
            'selection': {'filter': 'item', 'values': ['0']}   # 0 = Totalt
        },
        {
            'code': 'Tid',
            'selection': {'filter': 'all', 'values': ['*']}
        }
    ]
}

try:
    bpi_raw = ssb_query('07221', bpi_query)
    bpi_df = jsonstat2_to_df(bpi_raw)
    print('BPI kolonner:', bpi_df.columns.tolist())
    print(bpi_df.tail())
except Exception as e:
    print(f'Feil ved henting av BPI: {e}')
    # Fallback: tomme data
    bpi_df = pd.DataFrame()

In [ ]:
def parse_quarter(s):
    """Konverterer SSB-kvartalstreng '2010K1' til Pandas Period."""
    # SSB bruker format '2010K1', '2010K2' osv.
    s = str(s)
    if 'K' in s:
        year, q = s.split('K')
        return pd.Period(f'{year}Q{q}', freq='Q')
    return pd.NaT

if not bpi_df.empty:
    # Finn tidskolonnen
    time_col = [c for c in bpi_df.columns if 'id' in c.lower() or 'tid' in c.lower() or 'Tid' in c][-1]
    # Fallback: bruk siste ikke-value kolonne
    if 'Tid' in bpi_df.columns:
        time_col = 'Tid'

    bpi_df['period'] = bpi_df[time_col].apply(parse_quarter)
    bpi_series = bpi_df.set_index('period')['value'].sort_index()
    bpi_series.name = 'BPI'
    bpi_series = pd.to_numeric(bpi_series, errors='coerce').dropna()
    print(f'BPI: {bpi_series.index[0]} – {bpi_series.index[-1]}, {len(bpi_series)} observasjoner')
    print(bpi_series.tail())

## 2. Byggekostnadsindeks (tabell 08655)

In [ ]:
# Tabell 08655: Byggekostnadsindeks for bustadblokk (2015=100, månedlig)
bki_query = {
    'query': [
        {
            'code': 'Komponent',
            'selection': {'filter': 'item', 'values': ['00']}  # 00 = Totalindeks
        },
        {
            'code': 'Tid',
            'selection': {'filter': 'all', 'values': ['*']}
        }
    ]
}

try:
    bki_raw = ssb_query('08655', bki_query)
    bki_df = jsonstat2_to_df(bki_raw)
    print('BKI kolonner:', bki_df.columns.tolist())
    print(bki_df.tail())
except Exception as e:
    print(f'Feil ved henting av BKI (08655): {e}')
    # Prøv alternativ tabell 03014
    try:
        bki_raw = ssb_query('03014', bki_query)
        bki_df = jsonstat2_to_df(bki_raw)
        print('BKI (03014) kolonner:', bki_df.columns.tolist())
    except Exception as e2:
        print(f'Feil ved alternativ BKI: {e2}')
        bki_df = pd.DataFrame()

In [ ]:
def parse_month(s):
    """Konverterer SSB-månedstreng '2010M01' til Pandas Period."""
    s = str(s)
    if 'M' in s:
        year, m = s.split('M')
        return pd.Period(f'{year}-{m}', freq='M')
    return pd.NaT

if not bki_df.empty:
    if 'Tid' in bki_df.columns:
        bki_df['period_m'] = bki_df['Tid'].apply(parse_month)
    else:
        time_col = [c for c in bki_df.columns if c != 'value'][-1]
        bki_df['period_m'] = bki_df[time_col].apply(parse_month)

    bki_monthly = bki_df.set_index('period_m')['value'].sort_index()
    bki_monthly = pd.to_numeric(bki_monthly, errors='coerce').dropna()

    # Konverter til kvartalsvis (gjennomsnittsverdi per kvartal)
    bki_q = bki_monthly.groupby(bki_monthly.index.asfreq('Q')).mean()
    bki_q.name = 'BKI'
    print(f'BKI (kvartalsvis): {bki_q.index[0]} – {bki_q.index[-1]}, {len(bki_q)} observasjoner')
    print(bki_q.tail())

## 3. Igangsatte boliger (tabell 06265)

In [ ]:
# Tabell 06265: Igangsatte boliger etter boligtype (kvartalsvis)
ig_query = {
    'query': [
        {
            'code': 'Boligtype',
            'selection': {'filter': 'item', 'values': ['00']}  # 00 = Totalt
        },
        {
            'code': 'Tid',
            'selection': {'filter': 'all', 'values': ['*']}
        }
    ]
}

try:
    ig_raw = ssb_query('06265', ig_query)
    ig_df = jsonstat2_to_df(ig_raw)
    print('Igangsatte kolonner:', ig_df.columns.tolist())
    print(ig_df.tail())
except Exception as e:
    print(f'Feil ved henting av igangsatte (06265): {e}')
    # Prøv tabell 05432
    try:
        ig_raw = ssb_query('05432', ig_query)
        ig_df = jsonstat2_to_df(ig_raw)
        print('Igangsatte (05432) kolonner:', ig_df.columns.tolist())
    except Exception as e2:
        print(f'Feil: {e2}')
        ig_df = pd.DataFrame()

In [ ]:
if not ig_df.empty:
    if 'Tid' in ig_df.columns:
        ig_df['period'] = ig_df['Tid'].apply(parse_quarter)
    else:
        time_col = [c for c in ig_df.columns if c != 'value'][-1]
        ig_df['period'] = ig_df[time_col].apply(parse_quarter)

    ig_series = ig_df.set_index('period')['value'].sort_index()
    ig_series = pd.to_numeric(ig_series, errors='coerce').dropna()
    ig_series.name = 'Igangsatte'
    print(f'Igangsatte: {ig_series.index[0]} – {ig_series.index[-1]}, {len(ig_series)} observasjoner')
    print(ig_series.tail())

## 4. Beregn Tobin's Q

$$Q_t = \frac{\text{BPI}_t}{\text{BKI}_t}$$

Begge indekser er normalisert til **2015=100**, slik at Q=1 betyr at markedsprisen tilsvarer byggekostnaden i 2015-kroner.

In [ ]:
# Slå sammen til ett datasett
data = pd.DataFrame({
    'BPI': bpi_series,
    'BKI': bki_q,
    'Igangsatte': ig_series
}).dropna(subset=['BPI', 'BKI'])

# Beregn Tobin's Q
data['Q'] = data['BPI'] / data['BKI']

# Log-transformasjoner (for regresjonsmodellen)
data['log_Q']          = np.log(data['Q'])
data['log_Igangsatte'] = np.log(data['Igangsatte'].replace(0, np.nan))

# Q lagged 1 og 2 kvartal (byggeprosjekter tar tid)
data['Q_lag1'] = data['Q'].shift(1)
data['Q_lag2'] = data['Q'].shift(2)
data['log_Q_lag1'] = data['log_Q'].shift(1)
data['log_Q_lag2'] = data['log_Q'].shift(2)

print(f'Datasett: {data.index[0]} – {data.index[-1]}')
print(f'Antall observasjoner: {len(data)}')
data[['BPI', 'BKI', 'Q', 'Igangsatte']].describe().round(2)

## 5. Visualisering

In [ ]:
idx = data.index.to_timestamp()

fig, axes = plt.subplots(3, 1, figsize=(13, 11))

# Panel 1: Indekser
axes[0].plot(idx, data['BPI'], label='Boligprisindeks (BPI)', color='steelblue', lw=2)
axes[0].plot(idx, data['BKI'], label='Byggekostnadsindeks (BKI)', color='tomato', lw=2)
axes[0].axhline(100, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].set_title('Boligprisindeks vs. Byggekostnadsindeks (2015=100)', fontsize=13)
axes[0].set_ylabel('Indeks')
axes[0].legend()

# Panel 2: Tobin's Q
axes[1].plot(idx, data['Q'], color='seagreen', lw=2)
axes[1].axhline(1.0, color='black', lw=1.2, ls='--', label='Q = 1 (likevekt)')
axes[1].fill_between(idx, 1, data['Q'],
                     where=data['Q'] >= 1, alpha=0.15, color='seagreen', label='Q > 1 (lønnsomt å bygge)')
axes[1].fill_between(idx, 1, data['Q'],
                     where=data['Q'] < 1, alpha=0.15, color='tomato', label='Q < 1 (ulønnsomt)')
axes[1].set_title("Tobin's Q = BPI / BKI", fontsize=13)
axes[1].set_ylabel("Q")
axes[1].legend()

# Panel 3: Igangsatte boliger
if 'Igangsatte' in data.columns and data['Igangsatte'].notna().sum() > 0:
    axes[2].bar(idx, data['Igangsatte'], width=70, color='slateblue', alpha=0.7)
    axes[2].set_title('Igangsatte boliger (antall per kvartal)', fontsize=13)
    axes[2].set_ylabel('Antall boliger')
else:
    axes[2].text(0.5, 0.5, 'Igangsatte-data ikke tilgjengelig', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=12)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig('tobins_q_norway.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figur lagret som tobins_q_norway.png')

## 6. Stasjonaritetstest (ADF)

Tobin's Q-modeller estimeres ofte med **nivåvariabler** (ECM) eller **første differanser** (OLS). Vi tester først om seriene er stasjonære.

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'{name}:')
    print(f'  ADF-statistikk: {result[0]:.3f}')
    print(f'  p-verdi:        {result[1]:.4f}')
    print(f'  Kritisk (5%):   {result[4]["5%"]:.3f}')
    stasjonaer = result[1] < 0.05
    print(f'  Konklusjon:     {"Stasjonær" if stasjonaer else "Ikke-stasjonær (I(1)?)"}')
    print()
    return stasjonaer

print('=== ADF-test: Nivå ===')
q_stat    = adf_test(data['log_Q'], 'log(Q)')
ig_stat   = adf_test(data['log_Igangsatte'].dropna(), 'log(Igangsatte)')

print('=== ADF-test: Første differanse ===')
adf_test(data['log_Q'].diff().dropna(), 'Δlog(Q)')
adf_test(data['log_Igangsatte'].diff().dropna(), 'Δlog(Igangsatte)')

## 7. Regresjonsmodell: Igangsatte boliger ~ Tobin's Q

Vi estimerer to modeller:

**Modell 1 – Nivå (OLS):**
$$\log(H_t) = \alpha + \beta_1 \log(Q_{t-1}) + \beta_2 \log(Q_{t-2}) + \varepsilon_t$$

**Modell 2 – Første differanser (robusthet mot ikke-stasjonaritet):**
$$\Delta\log(H_t) = \alpha + \beta_1 \Delta\log(Q_{t-1}) + \beta_2 \Delta\log(Q_{t-2}) + \varepsilon_t$$

In [ ]:
reg_data = data[['log_Igangsatte', 'log_Q', 'log_Q_lag1', 'log_Q_lag2']].dropna()

if len(reg_data) < 10:
    print('For få observasjoner for regresjon – sjekk datahenting ovenfor.')
else:
    # ------- Modell 1: Nivå -------
    y1 = reg_data['log_Igangsatte']
    X1 = sm.add_constant(reg_data[['log_Q_lag1', 'log_Q_lag2']])
    mod1 = sm.OLS(y1, X1).fit(cov_type='HC3')  # Heteroskedastisitetsrobust SE

    print('=' * 60)
    print('MODELL 1: Nivå – log(Igangsatte) ~ log(Q_lag1) + log(Q_lag2)')
    print('=' * 60)
    print(mod1.summary())

In [ ]:
if len(reg_data) >= 10:
    # ------- Modell 2: Første differanser -------
    diff_data = reg_data.diff().dropna()

    y2 = diff_data['log_Igangsatte']
    X2 = sm.add_constant(diff_data[['log_Q_lag1', 'log_Q_lag2']])
    mod2 = sm.OLS(y2, X2).fit(cov_type='HC3')

    print('=' * 60)
    print('MODELL 2: Diff – Δlog(Igangsatte) ~ Δlog(Q_lag1) + Δlog(Q_lag2)')
    print('=' * 60)
    print(mod2.summary())

## 8. Elastisitet og tolkning

β-koeffisientene i log-log-modellen tolkes direkte som **elastisiteter**:
- β = 1.5 betyr at 1% økning i Q → 1.5% økning i igangsatte boliger.

In [ ]:
if len(reg_data) >= 10:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: log(Q_lag1) vs log(Igangsatte)
    ax = axes[0]
    ax.scatter(reg_data['log_Q_lag1'], reg_data['log_Igangsatte'],
               alpha=0.5, color='steelblue', s=40)
    x_range = np.linspace(reg_data['log_Q_lag1'].min(), reg_data['log_Q_lag1'].max(), 100)
    # Lag en enkel enkel-variabel OLS for scatter-linja
    simple = sm.OLS(reg_data['log_Igangsatte'],
                    sm.add_constant(reg_data['log_Q_lag1'])).fit()
    ax.plot(x_range, simple.params[0] + simple.params[1]*x_range,
            color='tomato', lw=2, label=f'OLS (β={simple.params[1]:.2f})')
    ax.set_xlabel("log(Q), lagged 1 kvartal")
    ax.set_ylabel("log(Igangsatte)")
    ax.set_title("Igangsatte vs. Tobin's Q")
    ax.legend()

    # Tidsserie: predikert vs faktisk (Modell 1)
    ax2 = axes[1]
    fitted_idx = reg_data.index.to_timestamp()
    ax2.plot(fitted_idx, np.exp(y1), label='Faktisk', color='steelblue', lw=1.5)
    ax2.plot(fitted_idx, np.exp(mod1.fittedvalues), label='Predikert (Modell 1)',
             color='tomato', lw=1.5, ls='--')
    ax2.set_title('Faktisk vs. predikert: igangsatte boliger')
    ax2.set_ylabel('Antall')
    ax2.legend()
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax2.xaxis.set_major_locator(mdates.YearLocator(3))

    plt.tight_layout()
    plt.savefig('model_fit.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figur lagret som model_fit.png')

## 9. Last inn dine egne data

Har du egne tall for BPI eller BKI kan du erstatte SSB-hentingen over med:

```python
# Eksempel: din_bpi.csv med kolonner 'kvartal' (f.eks. '2005Q1') og 'bpi'
din_bpi = pd.read_csv('din_bpi.csv', index_col='kvartal', parse_dates=True)
din_bpi.index = pd.PeriodIndex(din_bpi.index, freq='Q')
bpi_series = din_bpi['bpi']

# Lim deretter inn i data-rammeverket ovenfor og kjør cellen for Q-beregning.
```

## 10. Eksporter data til CSV

In [ ]:
export = data[['BPI', 'BKI', 'Q', 'Igangsatte']].copy()
export.index = export.index.to_timestamp()
export.index.name = 'kvartal'
export.to_csv('tobins_q_data.csv')
print('Data eksportert til tobins_q_data.csv')
export.tail(10)

---
## Metodeoppsummering

| | Detalj |
|---|---|
| **Q-definisjon** | BPI (brukte boliger) / BKI (boligblokk), begge 2015=100 |
| **Lag** | 1–2 kvartal (byggeprosjekter krever planlegging) |
| **Estimering** | OLS med HC3 standardfeil (robuste mot heteroskedastisitet) |
| **Tolkning** | Log-log → direkte elastisitetsestimater |
| **Dataperiode** | Ca. 2005–i dag (avhengig av SSB-tilgjengelighet) |

### Mulige utvidelser
- Inkluder **rente** (Norges Banks styringsrente) som kontrollvariabel  
- Bruk **feilkorreksjonsmodell (ECM)** hvis seriene er ko-integrerte  
- Del opp etter **boligtype** (enebolig vs. flerbolig)  
- Regional analyse (Oslo, Bergen, Stavanger)